# Index Raising & Lowering — Covariant and Contravariant Tensors

In general relativity, the metric tensor is used to raise and lower tensor indices:

- **Raise an index**: $T^a{}_b = g^{ac}\,T_{cb}$
- **Lower an index**: $T_a{}^b = g_{ac}\,T^{cb}$

Gravica provides `raise_index_2d` and `lower_index_2d` for rank-2 tensors.

In [1]:
import os
from symbolica import set_license_key

_key = os.environ.get("SYMBOLICA_LICENSE", "")
if _key:
    set_license_key(_key)

from gravica import (
    RicciTensor,
    EinsteinTensor,
    display,
    raise_index_2d,
    lower_index_2d,
)
from gravica.simplify import simplify, str_is_zero
from gravica.metrics import de_sitter

## 1. Ricci Tensor of de Sitter Spacetime

de Sitter spacetime is **not** a vacuum solution, so $R_{ab} \neq 0$.
This makes it a good testbed for index manipulation.

In [2]:
g = de_sitter()
coords = ["t", "r", "\\theta", "\\varphi"]

ricci = RicciTensor.from_metric(g)

print("R_{ab} (covariant):")
display.components_table(display.nonzero_components(ricci, coords), ricci)

R_{ab} (covariant):
┌────────────────────────────────────────────────────────┐
│ You are running a restricted Symbolica instance.       │
│                                                        │
│ This mode is only permitted for non-commercial use and │
│ is limited to one instance and core per machine.       │
│                                                        │
│ Hobbyists can easily acquire a free license key        │
│ that unlocks all cores and removes this banner:        │
│                                                        │
│   from symbolica import *                              │
│   request_hobbyist_license('YOUR_NAME', 'YOUR_EMAIL')  │
│                                                        │
│ All other users can obtain a free 30-day trial key:    │
│                                                        │
│   from symbolica import *                              │
│   request_trial_license('NAME', 'EMAIL', 'EMPLOYER')   │
│                                   

| Component | Value |
|-----------|-------|
| $R_{t t}$ | $\frac{1}{3} \left(-3 \Lambda+r^{2} \Lambda^{2}\right)$ |
| $R_{r r}$ | $\frac{-3 \Lambda}{-3+r^{2} \Lambda}$ |
| $R_{\theta \theta}$ | $r^{2} \Lambda$ |
| $R_{\varphi \varphi}$ | $r^{2} \Lambda \sin\!\left(\theta\right)^{2}$ |

## 2. Raising the First Index: $R^a{}_b = g^{ac}R_{cb}$

In [3]:
R_lower = ricci.components
R_mixed = raise_index_2d(g, R_lower, which=0)

items = []
for a in range(g.dim):
    for b in range(g.dim):
        val = R_mixed[a][b]
        if not str_is_zero(val):
            items.append((f"^{{{coords[a]}}}{{}}_{{{coords[b]}}}", val))

print("R^a_b (mixed):")
display.components_table(items, tensor_symbol="R", index_style="christoffel")

R^a_b (mixed):


| Component | Value |
|-----------|-------|
| $R^{t}{}_{t}$ | $-\Lambda$ |
| $R^{r}{}_{r}$ | $-\Lambda$ |
| $R^{\theta}{}_{\theta}$ | $-\Lambda$ |
| $R^{\varphi}{}_{\varphi}$ | $-\Lambda$ |

## 3. Round-Trip: Raise Then Lower

Lowering the raised index should recover the original $R_{ab}$.

In [4]:
R_recovered = lower_index_2d(g, R_mixed, which=0)

all_match = True
for a in range(g.dim):
    for b in range(g.dim):
        diff = simplify(R_recovered[a][b] - R_lower[a][b])
        if not str_is_zero(diff):
            all_match = False
            print(f"Mismatch at ({a},{b}): diff = {diff}")

if all_match:
    print("Verified: raise then lower recovers the original tensor.")

Verified: raise then lower recovers the original tensor.


## 4. Mixed-Index Einstein Tensor

The Einstein equations in mixed form $G^a{}_b$ are often diagonal, which reveals the physical content more clearly.

In [5]:
einstein = EinsteinTensor.from_metric(g)

print("G_{ab} (covariant):")
display.components_table(display.nonzero_components(einstein, coords), einstein)

G_mixed = raise_index_2d(g, einstein.components, which=0)

items = []
for a in range(g.dim):
    for b in range(g.dim):
        val = G_mixed[a][b]
        if not str_is_zero(val):
            items.append((f"^{{{coords[a]}}}{{}}_{{{coords[b]}}}", val))

print("\nG^a_b (mixed):")
display.components_table(items, tensor_symbol="G", index_style="christoffel")

G_{ab} (covariant):

G^a_b (mixed):


| Component | Value |
|-----------|-------|
| $G^{t}{}_{t}$ | $\Lambda$ |
| $G^{r}{}_{r}$ | $\Lambda$ |
| $G^{\theta}{}_{\theta}$ | $\Lambda$ |
| $G^{\varphi}{}_{\varphi}$ | $\Lambda$ |